# BMKMANU S1000 - exp74

## Import packages

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

In [ ]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _bmkmanu import bmkmanu

## Create sdata

In [ ]:
dataset_id = 'exp102'

path = r'D:\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\exp102\BSTMAtrix'
config_prefix = 'exp102_28Feb2025'
technology = 'S3000'
output_path = r'd:\2024-07-UTBenchmark-SpC\data\processed\BMK_S3000\exp102\harpy\sdata.zarr'

pipeline_name = 'BSTMAtrix'
species = 'mouse'
tissue_annotation_file = f'D:/2024-07-UTBenchmark-SpC/data/processed/BMK_S3000/{dataset_id}/tissue_annotation/tissue.geojson'


In [ ]:
sdata = bmkmanu(
    path=path, 
    dataset_id=dataset_id,
    filtered_counts=False,
    add_in_tissue_column=True,
    levels=None, 
    technology=technology,
    # output=output_path
)
sdata

# TODO: Remove or annotate empty spots

In [ ]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

## Specify plotting parameters

In [ ]:
DPI = 300
MAX_DIM_FOR_PLOTS = 5000

HE_image = f'{dataset_id}_he'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

## Plot H&E

In [ ]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

## Plot BSTMAtrix levels and tissue annotation

In [ ]:
for level in levels:
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} {level} - {pipeline_name} tissue annotation"
    
    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.1, contour_px=5, outline_alpha=1, color='in_tissue', scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_annotation_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

## Calculate pixel size

In [ ]:
# Determine pixel size in microns
import tifffile

path_img = r'd:\2024-07-UTBenchmark-SpC\data\raw\BMK_S1000\exp74\EXP74_BMK_WT_mousebrain_singles\EXP74_BMK_WT_mousebrain_singles\tile_x001_y001.tif'

with tifffile.TiffFile(path_img) as tif:
    xres = tif.pages[0].tags["XResolution"].value  # (numerator, denominator)
    yres = tif.pages[0].tags["YResolution"].value
    unit = tif.pages[0].tags["ResolutionUnit"].value

    px_per_cm_x = xres[0] / xres[1]
    px_per_cm_y = yres[0] / yres[1]

    if unit.name == "CENTIMETER":
        microns_per_pixel_x = 10_000 / px_per_cm_x
        microns_per_pixel_y = 10_000 / px_per_cm_y
    elif unit.name == "INCH":
        microns_per_pixel_x = 25_400 / px_per_cm_x
        microns_per_pixel_y = 25_400 / px_per_cm_y
    else:
        raise ValueError(f"Unknown ResolutionUnit: {unit}")

if microns_per_pixel_x == microns_per_pixel_y:
    microns_per_pixel = microns_per_pixel_x
    print(f"Pixel size: {microns_per_pixel:.3f} µm")
else:
    print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

In [ ]:
# Determine pixel size in microns
he_shape = hp.im._image._get_spatial_element(sdata, f'{dataset_id}_he').shape

he_width = he_shape[2]
he_height = he_shape[1]

microns_per_pixel_x = 6800 / he_shape[2]
microns_per_pixel_y = 6800 / he_shape[1]
print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

microns_per_pixel = (microns_per_pixel_x + microns_per_pixel_y) / 2
print(f"Pixel size: {microns_per_pixel:.3f} µm")

# NOTE: This is not a great way to have to summise what the pixel size is. This was recommended by BMKGENE and should be good enough, but there is definitely the possibility that the capture area is not exactly 6.8 micrometer.

## Create bounding box for total capture area

In [ ]:
# Add bounding box for total capture area
import shapely

geoms = sdata.shapes[f'{dataset_id}_level_1'].geometry
bbox = shapely.MultiPolygon(list(geoms)).minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1'], get_all=True),
    overwrite = True,
)

In [ ]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_level_1', color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.03)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

In [ ]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

## Add tissue annotation

In [ ]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[f'{dataset_id}_level_1'], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

In [ ]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

In [ ]:
# Determine which spots are in tissue annotation
for level in levels:
    sdata = add_in_tissue(
        sdata, 
        table_layer = level, 
        tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
    )

In [ ]:
# Check spots in tissue
for level in levels:
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = "Tissue annotation spots"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_{level}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue_annotation', table_layer=level, scale=best_scale
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.02)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'qupath_annotation_spots_{level}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

In [ ]:
# Total number of spots in table
n_spots = len(sdata.tables['level_1'].obs)
print(f"Number of spots in table: {n_spots}") # NOTE: This contains a lot of false spots as well (i.e. spots that are not actually on the capture slide)

In [ ]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(sdata.tables['level_1'].obs[sdata.tables['level_1'].X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

In [ ]:
# Number of spots in tissue
n_spots_in_tissue = len(sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

In [ ]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = sdata.tables['level_1'].obs['in_tissue_annotation'] == 1
umis_mask = np.array(sdata.tables['level_1'].X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero}")

In [ ]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

## Capture effeciency metrics

In [ ]:
# Total number of genes
print(f"Total number of genes: {(sdata.tables['level_1'].X.sum(axis=0) > 0).sum()}")

In [ ]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].X.sum(axis=0) > 0).sum()}")

In [ ]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables['level_1'].X.sum()}")

In [ ]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1].X.sum()}")

In [ ]:
# TODO: Take into account empty spots

In [ ]:
umi_gene = pd.read_csv(os.path.join(path, config_prefix, 'umi_gene.tsv.gz'), sep='\t', header=None)
umi_gene

In [ ]:
# Total number of UMIs
print("Total number of UMIs:", umi_gene[3].sum())

In [ ]:
# Total number of unique barcodes
print("Total number of unique barcodes:", umi_gene[0].nunique())

In [ ]:
barcode_pos = pd.read_csv(os.path.join(path, config_prefix, 'barcode_pos.tsv'), sep='\t', header=None)
barcode_pos

In [ ]:
barcode = pd.read_csv(os.path.join(path, config_prefix, 'barcode.tsv'), sep='\t', header=None)
barcode

In [ ]:
barcodes_read = pd.read_csv(r'd:\2024-07-UTBenchmark-SpC\data\processed\BMK_S1000\exp74\BSTMAtrix\05.AllheStat\BSTViewer_project\level_matrix\level_1\barcodes_read.tsv.gz', compression='gzip', sep='\t', header=None)
barcodes_read

In [ ]:
barcodes_read = pd.read_csv(r'd:\2024-07-UTBenchmark-SpC\data\processed\BMK_S1000\exp74\BSTMAtrix\05.AllheStat\BSTViewer_project\level_matrix\level_1\barcodes_read.tsv.gz', compression='gzip', sep='\t', header=None)
barcodes_read

In [ ]:
features = pd.read_csv(os.path.join(path, config_prefix, 'features.tsv'), sep='\t', header=None)
features

## Export unique UMI counts per gene

In [ ]:
adata_in_tissue = sdata.tables['level_1'][sdata.tables['level_1'].obs['in_tissue_annotation'] == 1]

gene_counts = np.array(adata_in_tissue.X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": adata_in_tissue.var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / overlap_area_mm2
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)

## QC

In [ ]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    sdata = qc_metrics(
        sdata,
        level,
        species
    )

In [ ]:
# Plot QC metrics
for level in levels:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        values = sdata.tables[level].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = qc_metric

        sdata.pl.render_images(
            element=HE_image, scale=best_scale
        ).pl.render_labels(
            element=f"{dataset_id}_{level}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=qc_metric, scale=best_scale, table_name=level, cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(plots_path, f'{qc_metric}_{level}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)